# **SIMT Transpose算子优化实践**

## 概述

本小节介绍 SIMT 编程模式下的 Transpose（矩阵转置）算子开发。Transpose 用于交换二维矩阵的行和列。

本节按照完整的逐步优化思路展开，逐一验证每一步优化的实际收益：

1. **直接基于 GM 读写实现 Transpose**：暴露转置写地址不连续、以及 Thread Block 超发带来的性能问题。
2. **限制 Thread Block 数为物理核数**：仍然直接读写 GM，先单独解决 Thread Block 超发问题，衡量这一步单独带来的收益。
3. **引入 UB 中转**：把非连续访问转移到 UB 内部，重点对比两种不同的 Thread Block 内部处理方式（固定物理核数+线程数为2048、固定物理核数+线程数为1024），说明寄存器溢出会带来什么开销。
4. **优化 UB Bank 冲突**：通过 UB padding 消除转置读阶段的 bank 冲突。
5. **双缓冲流水并行**：让相邻迭代的 GM 访问与 UB 内部转置重叠执行；并在这一步之后补充一个纯连续读写的 GM 带宽基线，衡量当前硬件在不做任何转置计算时的访存效率上限，作为整条优化路径的参照。

每一步都会给出真实编译运行后采集到的性能数据，用于验证该优化点是否达到了预期效果。

### 学习前置要求

学习本小节前，建议已经具备以下基础：

- 已学习《SIMT编程模型》中核函数、线程索引、线程组织、内存层级等内容。
- 已学习 Gather 算子编程实战，了解 `blockIdx`、`threadIdx`、`blockDim`、`gridDim` 的基本使用方式。
- 了解基本的 Ascend C SIMT 算子开发和执行流程。

### 学习目标

完成本小节后，开发者应能够：

- 理解 GM 直接转置为什么会出现连续读、非连续写的访存模式，并知道如何用一个纯连续拷贝的基线核函数衡量 GM 带宽上限。
- 学会如何通过引入 UB 中转优化 Transpose 的 GM 访问模式，使全局内存读写保持连续访问。
- 理解 Thread Block 启动数量与硬件物理核数之间的关系：既要避免 Thread Block 超发导致的调度排队开销，也要避免单个 Thread Block 处理过多线程导致的寄存器溢出。
- 掌握 UB bank 的基本排布方式，以及如何通过 padding 缓解 bank 冲突。
- 理解双缓冲（ping/pong）如何让 GM 访问与 UB 内部计算相互重叠，进一步压缩执行耗时。
- 能够使用 `msOpProf` 采集并解读每一步优化对应的真实性能数据。

### 本节内容

- 环境准备
- Transpose 算子功能介绍
- Transpose 算子实现
- 小结


## 1. 环境准备

正式开始学习之前，先执行下方脚本检查 CANN Toolkit 是否可用，并把 CANN 环境变量加载到当前 Jupyter 进程，保证后续能够正常导入相关代码并使用 bisheng 编译器完成算子的开发与编译。

本节所有编译、运行和练习修改都在 `Sources/07.05` 目录下进行，`src` 目录仅作为只读的源码仓库存放原始代码。


In [ ]:
import os
import subprocess
import shlex
from pathlib import Path


def find_cann_home():
    candidates = []
    for key in ["ASCEND_HOME_PATH", "ASCEND_TOOLKIT_HOME"]:
        value = os.environ.get(key)
        if value:
            candidates.append(Path(value).expanduser())

    candidates.extend([
        Path.home() / "Ascend/cann",
        Path.home() / "Ascend/ascend-toolkit/latest",
        Path("/usr/local/Ascend/cann"),
        Path("/usr/local/Ascend/ascend-toolkit/latest"),
    ])

    for candidate in candidates:
        normalized = candidate
        if normalized.name in {"x86_64-linux", "aarch64-linux"}:
            normalized = normalized.parent
        set_env = normalized / "set_env.sh"
        if set_env.exists():
            return normalized.resolve(), set_env.resolve()

    raise RuntimeError("未找到 CANN Toolkit，请确认已安装 CANN，并设置环境变量。")


def source_cann_env(set_env):
    command = f"set -a && source {shlex.quote(str(set_env))} >/dev/null 2>&1 && env"
    result = subprocess.run(["bash", "-lc", command], check=True, text=True, capture_output=True)
    for line in result.stdout.splitlines():
        if "=" in line:
            key, value = line.split("=", 1)
            os.environ[key] = value


cann_home, cann_set_env = find_cann_home()
source_cann_env(cann_set_env)

WORKSPACE = Path("Sources/07.05")
WORKSPACE.mkdir(parents=True, exist_ok=True)

print(f"CANN Toolkit: {cann_home}")
print(f"Workspace: {WORKSPACE.resolve()}")


## 2. Transpose 算子功能介绍

Transpose 用于交换二维矩阵的行和列，计算公式如下：

```text
output(col, row) = input(row, col)
```

实际计算中，`input` 和 `output` 都按一维数组存储。设输入矩阵形状为 `height x width`，对于输入矩阵中的 `input(row, col)`，其一维下标为：

```cpp
input_index = row * width + col;
```

转置后，该元素写入输出矩阵的 `output(col, row)`，对应的一维下标为：

```cpp
output_index = col * height + row;
```

下图以 `4 x 3` 输入矩阵为例，展示 Transpose 后行列维度和元素排列的变化：

![](images/07_05_simt_transpose/transpose_example.png)

本节实现中，输入矩阵的 `height` 和 `width` 均为 `1024`。从最朴素实现到后续一系列性能优化路径如下所示：

| 核函数名 | 说明 |
| --- | --- |
| `transpose_gm_custom` | 直接基于 GM 读写实现转置，每个线程处理一个元素 |
| `transpose_gm_core_limited_custom` | 在 `transpose_gm_custom` 基础上增加核内循环，Thread Block 数固定为物理核数 |
| `transpose_ub_2tile_core_limited_custom` | UB 中转，Thread Block 数固定为物理核数，每个 Thread Block 处理两个 tile |
| `transpose_ub_fixed64_custom` | UB 中转，Thread Block 数固定为物理核数，每个 Thread Block 每次迭代处理一个 tile |
| `transpose_ub_padding_custom` | UB 中转 + padding，消除转置读的 bank 冲突 |
| `transpose_ub_padding_db_custom` | UB 中转 + padding + 双缓冲 |

下面，我们将一一对以上实现展开学习。

## 3. Transpose 算子实现

### 3.1 准备源码工作目录

完成环境准备并明确 Transpose 算子功能后，我们开始动手实践。本节按照完整的优化路径，依次实现下面几个版本，分别放在独立目录中：

- `gm`：直接基于 GM 读写实现 Transpose
- `gm_core_limited`：在 `gm` 的基础上限制 Thread Block 数为物理核数
- `ub_2tile_core_limited`：UB 中转，Thread Block 数固定为物理核数，每个 Thread Block 处理两个 tile
- `ub`：UB 中转，Thread Block 数固定为物理核数，每个 Thread Block 每次迭代处理一个 tile
- `ub_padding`：在 `ub` 的基础上引入 UB padding，消除 bank 冲突
- `ub_padding_db`：在 `ub_padding` 的基础上引入双缓冲
- `copy_baseline`：GM 带宽基线（纯连续拷贝），在双缓冲实现之后作为额外对比给出

上述版本的完整源码都已经准备在只读目录 `src/simt_transpose/` 下。执行下面的单元格，把这些源码拷贝到 `Sources/07.05/simt_transpose/` 作为本节的工作目录，后续的编译、运行和修改都只发生在 `Sources` 下：


In [ ]:
import shutil
from pathlib import Path

SRC_ROOT = Path("src/simt_transpose")            # 只读源码目录
DST_ROOT = Path("Sources/07.05/simt_transpose")  # 工作目录

VERSIONS = [
    "gm",
    "gm_core_limited",
    "ub_2tile_core_limited",
    "ub",
    "ub_padding",
    "ub_padding_db",
    "copy_baseline",
]

# 清理旧的工作目录，保证每次都从 src 拷贝出干净的一份
if DST_ROOT.exists():
    shutil.rmtree(DST_ROOT)

for version in VERSIONS:
    dst = DST_ROOT / version
    dst.mkdir(parents=True, exist_ok=True)
    for pattern in ("*.asc", "*.h", "CMakeLists.txt"):
        for f in sorted((SRC_ROOT / version).glob(pattern)):
            shutil.copy2(f, dst / f.name)
    print(f"{version}: {sorted(p.name for p in dst.iterdir())}")


### 3.2 直接基于 GM 读写实现 Transpose


#### 3.2.1 实现思路与代码

作为最简单的实现，本节让每个线程只处理输入矩阵中的一个元素，每个线程根据自己的全局下标 `idx` 直接算出在输入矩阵中的行列坐标，再根据转置映射计算输出地址并写入结果：

```cpp
uint32_t idx = blockIdx.x * blockDim.x + threadIdx.x;
uint32_t row = idx / width;
uint32_t col = idx - row * width;
// input(row,col) -> output(col,row)
output[col * height + row] = input[idx];
```


输入矩阵共有 `1024 * 1024` 个元素，由于计算复杂度低，每个 Thread Block 可启动 `2048` 个线程，让线程尽量并行起来，因此需要启动的 Thread Block 数为：

```cpp
uint32_t num_blocks = input_total_length / THREAD_COUNT; // 1024 * 1024 / 2048 = 512
```

对应的启动配置如下：

```cpp
dim3 grid(512, 1, 1);
dim3 block(2048, 1, 1);
```

完整实现代码已保存在 `Sources/07.05/simt_transpose/gm/transpose_gm.asc`，执行下面的单元格查看完整源码：

In [ ]:
!cat Sources/07.05/simt_transpose/gm/transpose_gm.asc

#### 3.2.2 CMake 配置

对应的 `CMakeLists.txt` 内容如下(源码文件为`Sources/07.05/simt_transpose/gm/CMakeLists.txt`)：


```cmake
cmake_minimum_required(VERSION 3.16)

set(CMAKE_ASC_ARCHITECTURES "dav-3510" CACHE STRING "NPU ARCH, e.g. dav-3510")

find_package(ASC REQUIRED)
project(simt_transpose_gm LANGUAGES ASC CXX)

add_executable(demo
    transpose_gm.asc
)

target_compile_options(demo PRIVATE
    $<$<COMPILE_LANGUAGE:ASC>:--npu-arch=${CMAKE_ASC_ARCHITECTURES} --enable-simt>
)
```

**编译选项说明：**

| 选项 | 说明 |
| --- | --- |
| `--npu-arch=dav-3510` | 指定 NPU 架构版本，`dav-` 后为架构号，Ascend 950PR/Ascend 950DT 对应 `dav-3510` |
| `--enable-simt` | **启用 SIMT 编程场景**，编译 SIMT 算子必须添加 |


#### 3.2.3 编译运行并采集性能

执行以下命令编译并运行当前实现：


In [ ]:
!cd Sources/07.05/simt_transpose/gm && mkdir -p build && cd build && \
 cmake -DCMAKE_ASC_ARCHITECTURES=dav-3510 .. && make -j && \
 ./demo


编译运行成功后，若看到以下输出，则说明计算结果与预期完全一致：

```text
[Success] Case accuracy verification passed.
```

完成正确性验证后，使用 `msOpProf` 工具采集算子性能：


In [ ]:
!cd Sources/07.05/simt_transpose/gm/build && msopprof ./demo

本实现在 Ascend 950 环境、CANN 9.1.0 上实测的结果如下：

| 阶段 | 核函数 | Task Duration(μs) | Block Dim | aiv_vec_time(μs) |
| --- | --- | --- | --- | --- |
| 直接读写 GM（Thread Block 超发） | `transpose_gm_custom` | 57.37 | 512 | 6.12 |

`aiv_vec_time` 只有 6.12μs，但 Task Duration 却高达 57.37μs，`aiv_vec_time`统计的是每个线程块的Vec操作耗时，两者差异大的主要原因在于：本实现按"一个线程处理一个元素"的最简单方式启动，Thread Block 数直接等于 `总元素数 / 每 Block 线程数 = 512`，远超设备物理核数（64），因此一个物理核平均要串行处理512/64 = 8个线程块任务。

`aiv_vec_time`统计的是每个线程块的平均AIV耗时，那么每个物理核实际的端到端`aiv_vec_time`约等于6.12μs * 8 = 48.96μs。超发的 Thread Block 需要排队等待前面的执行完成才能调度，每次调度都会产生额外的头尾开销，从上述数据大约可计算出整体头尾开销约为57.37 - 48.96 = 8.4μs，说明这种质朴写法虽然简单，但确引入较重的头尾开销，性能差。

从当前数据看，启动512个线程块引入的开销极大，下面优先解决超发问题。

### 3.3 限制 Thread Block 数为物理核数

#### 3.3.1 实现思路与代码

本实现将在上一实现的基础上优化线程块数量，实现很简单：保持 `transpose_gm_custom` 直接读写 GM 的坐标计算不变，只是不再让 Thread Block 数直接等于总元素数除以每 Block 线程数，而是限制在物理核数，核内增加一层 grid-stride for 循环，让每个 Thread Block 循环处理多组元素：

```cpp
uint32_t stride = gridDim.x * blockDim.x;
for (uint32_t idx = blockIdx.x * blockDim.x + threadIdx.x; idx < total_elements; idx += stride) {
    uint32_t row = idx / width;
    uint32_t col = idx - row * width;
    output[col * height + row] = input[idx];
}
```

封装 `get_vector_core_num()` 接口，在host侧通过 `aclrtSetDevice` 接口获取当前设备的实际物理核数。封装的 `get_vector_core_num()`实现如下：

```cpp
// 查询硬件 vector core 数（AIV 核数），不同环境上的物理核数不同，需要在运行时获取
uint32_t get_vector_core_num(uint32_t device_id)
{
    int64_t core_num = 0;
    aclError ret = aclrtGetDeviceInfo(device_id, ACL_DEV_ATTR_VECTOR_CORE_NUM, &core_num);
    if (ret != ACL_SUCCESS) {
        return 0;
    }
    return static_cast<uint32_t>(core_num);
}
```


启动配置从 `dim3 grid(512, 1, 1)` 改为按查询结果设置：

```cpp
uint32_t num_blocks = get_vector_core_num(device_id);
dim3 grid(num_blocks, 1, 1);
dim3 block(2048, 1, 1);
```

这一步没有改变任何 GM 访问坐标，转置写回依然是跨行、非连续的；唯一的变化是 Thread Block 数从 512 降到物理核数（测试机器上为 64），核内循环覆盖原来由多个 Thread Block 分担的工作量。完整实现代码与 CMakeLists.txt 已保存在 `Sources/07.05/simt_transpose/gm_core_limited/` 目录下（核心逻辑如上），此处不再重复展示全文。

In [ ]:
!cat Sources/07.05/simt_transpose/gm_core_limited/gm_core_limited.asc

#### 3.3.2 编译运行并采集性能

完成 CMake 配置后，执行以下命令编译并运行当前实现：

In [ ]:
!cd Sources/07.05/simt_transpose/gm_core_limited && mkdir -p build && cd build && \
 cmake -DCMAKE_ASC_ARCHITECTURES=dav-3510 .. && make -j && \
 ./demo

In [ ]:
!cd Sources/07.05/simt_transpose/gm_core_limited/build && msopprof ./demo

本实现在 Ascend 950 环境、CANN 9.1.0 上实测的结果如下：

| 阶段 | 核函数 | Task Duration(μs) | Block Dim | aiv_vec_time(μs) |
| --- | --- | --- | --- | --- |
| 直接读写 GM（Thread Block 超发） | `transpose_gm_custom` | 57.37 | 512 | 6.12 |
| 直接读写 GM（限制核数为 64） | `transpose_gm_core_limited_custom` | 35.674 | 64 | 32.899 |

`Block Dim` 从 512 降为 64 后，Task Duration 明显下降（57.37μs → 35.674μs），说明消除 Thread Block 超发确实带来了收益；但 `aiv_vec_time` 却大幅上升（6.12μs → 32.899μs）。原因是：转置写回阶段跨行写入导致 GM 写不连续，同一 Warp 内相邻线程的写地址被分散到输出矩阵的不同行，这部分开销在上一实现中被线程块切换的头尾开销所掩盖，没有体现在 `aiv_vec_time` 耗时上；现在 64 个 Thread Block 各自循环处理 8 组元素，同样的非连续写开销被均匀摊到了每次循环迭代中，因此完整地反映在了 `aiv_vec_time` 里。也就是说，这一步只是把"调度排队开销"转化成了"暴露出来的访存开销"，两者此消彼长，Task Duration 的下降幅度小于预期，非连续访问的问题并没有解决。

下一步将尝试引入 UB 中转，把这部分非连续访问转移到访问效率更高的 UB 内部，解决 GM 非连续访问带来的性能损耗。

### 3.4 引入 UB 中转优化 Transpose


#### 3.4.1 UB 中转的基本思路

3.3 节已经把 Thread Block 数固定为物理核数，消除了调度排队开销，但没有解决非连续访问本身的问题，因为直接基于 GM 读写时，输入侧读取是连续的，但转置写回输出矩阵时会沿列方向跨行写入，GM 写地址不连续。为了改善写回阶段的访问模式，本实现引入 UB 作为中间缓存，并把 `1024 x 1024` 输入矩阵划分为多个 `32 x 32` tile。这样可以以 tile 为单位组织转置：输入矩阵中的一个 tile 先按行方向连续读入 UB，完成 tile 内转置后，再写回到输出矩阵中对应的 tile 位置。

这里选择 tile 块单元大小为 `32 x 32`，主要是因为WarpSize是32，能够保证一个Warp内访存合并，同时便于用 tile 内的行列坐标描述线程负责的数据位置。对于输入为 `1024 x 1024` 矩阵，行方向和列方向各有 `32` 个 tile，因此一共有 `32 * 32 = 1024` 个 tile块。

在代码中，所有 tile 按行优先顺序编号。设 `tiles_per_row = width / tile_dim(32)`，则一维 `tile_id` 与 tile 的二维坐标关系为：

```cpp
uint32_t tile_row = tile_id / tiles_per_row;
uint32_t tile_col = tile_id - tile_row * tiles_per_row;
```

在单个 tile 内，使用 `local_row` 表示 tile 内行号，使用 `local_col` 表示 tile 内列号。结合 tile 的二维坐标后，当前元素在输入矩阵中的全局坐标为：

```cpp
uint32_t input_row = tile_row * tile_dim + local_row;
uint32_t input_col = tile_col * tile_dim + local_col;
```

转置后，输入 tile `(tile_row, tile_col)` 会写入输出矩阵中的 tile `(tile_col, tile_row)`。写回时仍按输出 tile 的行方向组织访问，其全局坐标关系为：

```cpp
uint32_t output_row = tile_col * tile_dim + local_row;
uint32_t output_col = tile_row * tile_dim + local_col;
```

下图展示了 UB 中转转置的数据流向。

<img src="./images/07_05_simt_transpose/transpose_ub_dataflow.png" alt="transpose_ub_dataflow"  width="700px" >

从图中可以看到，线程先按输入矩阵原布局把一个 tile 连续读入 UB；同步后，再从 UB 中按转置方向读取，并按输出矩阵行方向连续写回 GM。这样访问模式从 `GM 连续读 + GM 非连续写` 变为 `GM 连续读 + UB 转置方向读 + GM 连续写`。换句话说，这个优化并没有消除所有不连续访问，而是把不连续访问从 GM 转移到了 UB 内部，保障GM的访问都是连续的，因为UB的访问效率较高，非连续访问的性能影响比GM的小很多。

我们在[SIMT内存介绍课程](../03_programming_model/03.04.04_simt_memory_hierarchy.ipynb)中已经学习到，配置的最大线程数决定了每个线程拥有的寄存器个数。引入 UB 中转后，线程数越多，处理的 tile 越多，索引变量和分支判断增多，容易导致寄存器溢出；处理太少则会影响并发效率，所以还需要确定每个 Thread Block 处理启动多少线程数比较合适。下面先尝试启动2048个线程，若存在寄存器溢出问题，再逐步减少线程数。

#### 3.4.2 2048线程版本的核心代码实现

由于线程数是 2048，所以每个线程块能完成 2 个 tile 块的计算任务：`2048 / (32 * 32) = 2`，为理解方便，我们将这 2 个 tile 块组合成一个 Tile 组（`TILES_PER_BLOCK = 2`）。延续上一实现，把线程块数固定为运行时查询到的硬件物理核数（`get_vector_core_num()`，测试机器上为 64）来抑制头开销，因此每个线程块通过循环处理多组 tile：

```cpp
constexpr uint32_t TILES_PER_BLOCK = 2;
uint32_t num_blocks = get_vector_core_num(device_id); // 固定为物理核数，运行时查询
dim3 grid(num_blocks, 1, 1);
dim3 block(32, 32 * TILES_PER_BLOCK, 1); // 2048 个线程

for (uint32_t tile_base = blockIdx.x * TILES_PER_BLOCK; tile_base < total_tiles;
     tile_base += gridDim.x * TILES_PER_BLOCK) {
    uint32_t tile_id = tile_base + local_tile;
    // ... 加载、同步、转置写回、同步 ...
}
```

以 `blockIdx.x = 0` 为例，它依次处理 tile `(0,1), (128,129), (256,257), ...`；64 个 Thread Block 协同覆盖全部 1024 个 tile，每个 Thread Block 需要循环 8 次。循环体内的两次 `asc_syncthreads()` 分别起到不同作用：第一次同步确保 Thread Block 内所有线程都完成 tile 数据加载后，才能开始按转置方向读取 UB；第二次同步确保当前 tile 的转置写回全部完成后，才能进入下一轮循环开始加载新数据，避免新一轮加载覆盖还未写出的 UB 内容。

完整实现代码与 CMakeLists.txt 已保存在 `Sources/07.05/simt_transpose/ub_2tile_core_limited/` 目录下，此处不再重复展示全文。

In [ ]:
!cat Sources/07.05/simt_transpose/ub_2tile_core_limited/ub_2tile_core_limited.asc

**CMake 配置说明：**

由于本实现引入 UB 后计算复杂度增加不少，需要关注寄存器是否充足，因此 `CMakeLists.txt` 中额外添加了 `--cce-res-usage` 编译选项，用于在编译日志中输出寄存器和栈使用信息。完整实现代码与 CMakeLists.txt 已保存在 `Sources/07.05/simt_transpose/ub_2tile_core_limited/` 目录下。

编译运行并采集性能：

In [ ]:
!cd Sources/07.05/simt_transpose/ub_2tile_core_limited && mkdir -p build && cd build && \
 cmake -DCMAKE_ASC_ARCHITECTURES=dav-3510 .. && make -j && \
 ./demo && msopprof ./demo


编译日志中会打印类似下面的信息：

```text
[BISHENG] Function properties for _Z38transpose_ub_2tile_core_limited_customILj32EEvPfPKfjjj_simt_entry: Stack size: 24 bytes, Used register number: 16
```

`register number: 16` 说明寄存器已经用满，而 `Stack size: 24 bytes` 说明栈空间使用较多，当前核函数应该出现了寄存器溢出（spill）。

本实现在 Ascend 950 环境、CANN 9.1.0 上实测的结果如下：

| 阶段 | 核函数 | Task Duration(μs) | Block Dim | aiv_vec_time(μs) |
| --- | --- | --- | --- | --- |
| 直接读写 GM（限制核数为 64） | `transpose_gm_core_limited_custom` | 35.674 | 64 | 32.899 |
| UB 中转（固定物理核数，2 tile/block） | `transpose_ub_2tile_core_limited_custom` | 27.08 | 64 | 24.54 |

两者 Block Dim 相同，都是 64，因此可以直接对比引入 UB 中转本身带来的收益：Task Duration 从 35.674μs 降到 27.08μs，`aiv_vec_time` 从 32.899μs 降到 24.54μs，说明把非连续访问转移到 UB 内部确实降低了访存开销。但编译日志揭示了本实现造成寄存器溢出：每个 Thread Block 同时维护两个 tile 需要更多索引变量和分支判断，超出了可用寄存器数量，多余变量换出到栈上，抵消了一部分 UB 中转带来的收益。回顾前面提到的关系：配置的最大线程数越多，每个线程可用的寄存器就越少，因此下一步尝试降低最大线程数、提升每个线程拥有的寄存器个数，看能否消除寄存器溢出，进一步释放 UB 中转的潜力。需要注意，"降低线程数换取寄存器"并不是一种通用的优化手段，实际调优时仍需结合具体核函数的计算复杂度，通过实测在两者之间找到平衡点。


#### 3.4.3 降低最大线程数避免寄存器溢出

最大线程数从 `2048` 降为 `1024`：`__launch_bounds__(1024)`, 正好覆盖一个 `32 x 32` tile，每个 Thread Block 同时维护的 tile 数从 2 降为 1。启动的核数继续保持为运行时查询到的物理核数，核内循环逐个处理 tile：

```cpp
constexpr uint32_t MAX_THREAD_COUNT = 1024;
dim3 grid(num_blocks, 1, 1); // num_blocks = get_vector_core_num(device_id)
dim3 block(32, 32, 1); // 1024 个线程，一次只处理一个 tile

for (uint32_t tile_id = blockIdx.x; tile_id < total_tiles; tile_id += gridDim.x) {
    // ... 加载、同步、转置写回、同步 ...
}
```


核心数据路径如下：


```cpp
uint32_t input_row = tile_row * tile_dim + local_row;
uint32_t input_col = tile_col * tile_dim + local_col;
tile[local_row][local_col] = input[input_row * width + input_col];
asc_syncthreads();

uint32_t output_row = tile_col * tile_dim + local_row;
uint32_t output_col = tile_row * tile_dim + local_col;
output[output_row * height + output_col] = tile[local_col][local_row];
asc_syncthreads();
```

读入 UB 时，同一个 Warp 访问输入矩阵同一行的连续 32 个 `float`，GM 读是连续的。写回 GM 时，同一个 Warp 写入输出矩阵同一行的连续 32 个 `float`，GM 写也为连续访问。中间的 `tile[local_col][local_row]` 是 UB 内部的转置方向读取。相比上一步，启动的线程数从2048降为1024，每个线程可用寄存器从16增加到32。而且每个线程块处理的数据量刚好为32*32（1个tile块），索引变量和分支判断更少（不再需要 `local_tile` 区分 Thread Block 内的两个 tile），计算复杂度降低，预期能够消除寄存器溢出。完整实现代码与 CMakeLists.txt 已保存在 `Sources/07.05/simt_transpose/ub/` 目录下。


编译运行并采集性能：

In [ ]:
!cd Sources/07.05/simt_transpose/ub && mkdir -p build && cd build && \
 cmake -DCMAKE_ASC_ARCHITECTURES=dav-3510 .. && make -j && \
 ./demo && msopprof ./demo


本实现在 Ascend 950 环境、CANN 9.1.0 上实测的结果如下：

| 阶段 | 核函数 | Task Duration(μs) | Block Dim | aiv_vec_time(μs) |
| --- | --- | --- | --- | --- |
| UB 中转（固定物理核数，最大线程数为2048） | `transpose_ub_2tile_core_limited_custom` | 27.08 | 64 | 24.54 |
| UB 中转（固定物理核数，最大线程数为1024） | `transpose_ub_fixed64_custom` | 25.70 | 64 | 24.01 |

相比上一步，启动的线程数从2048降为1024，编译日志中的 Used register number 从 16 变为 24、Stack size 从 24 字节降为 0 字节,寄存器溢出被消除。Task Duration 相比上一步继续下降（27.08μs → 25.70μs），`aiv_vec_time` 也小幅下降（24.54μs → 24.01μs），由此可知，引入 UB 后需要调整最大线程数。正是因为 `1024` 线程正好覆盖一个 `32 x 32` tile，索引关系简单、寄存器压力更低；`2048` 线程需要一个 Thread Block 同时处理两个 tile，会引入更多索引变量、分支和三维 UB 访问，容易导致寄存器溢出，反而抵消了减少调度开销带来的收益。

不过，`aiv_vec_time` 仍然占了 Task Duration 的大部分（24.01μs / 25.70μs），说明当前的瓶颈已经从"Thread Block 调度"转移到了"UB 内部访问效率"。下一步分析并优化 UB 转置读取阶段的 bank 冲突。


### 3.5 优化 UB Bank 冲突

#### 3.5.1 UB bank 结构与 bank 冲突原理

Ascend 950PR/Ascend 950DT 的 UB 在物理上划分为 16 个 bank，每 2 个 bank 组成一个 bank group（共 8 个 bank group）；在 SIMT 编程模式下，每个 bank 又进一步划分为 4 个 subbank，即整个 UB 共 64 个 subbank：

<img src="./images/07_05_simt_transpose/bank_structure.png" alt="bank_structure"  width="1500px" >

当同一个 Warp 内的多个线程在同一条访存指令中，命中同一个 bank group 内编号相同的 subbank 时，硬件需要把这些访问串行化处理，由此产生的额外延迟称为 bank 冲突（更准确地说是 subbank 冲突）。根据访问类型不同，bank 冲突可以分为写写冲突和读读冲突两种。

回顾 3.4 节的实现，UB 中的 `tile` 数组按紧凑的 `32 x 32` 布局存放，每行 32 个 `float` 共 `32 * sizeof(float) = 128` 字节，恰好跨越 4 个 bank。按照 UB 的地址低位交织规则，`tile` 的第 1 行覆盖 bank0~bank3，第 2 行覆盖 bank4~bank7，第 3 行覆盖 bank8~bank11，其余行依次类推，每 4 行回到 bank0。转置读取 `tile[local_col][local_row]` 时，同一个 Warp 内 32 个线程的 `local_row` 相同、`local_col` 从 0 到 31 连续变化，这等价于读取 UB tile 的同一列，访问的 32 个地址依次相差 128 字节。由于行跨度固定为 32 个 `float`，这 32 次访问会集中落到两个 bank group 的 subbank 0 上，属于读读冲突，如下图所示：

<img src="./images/07_05_simt_transpose/case2_bank.png" alt="case2_bank"  width="1500px" >

要打破这种集中映射，需要改变 `tile` 每行的物理跨度，让同一列的相邻元素错开到不同的 subbank 上。本节把 `tile` 的行跨度从 32 增加到 34（即每行增加 2 列 padding），每行变为 34 个 `float`、共 `34 * sizeof(float) = 136` 字节，行跨度也就从 16 个 subbank 变为 17 个 subbank。17 是奇数，不再与 bank/subbank 的排布周期对齐，因此同一列的 32 个元素会依次错开排布到不同的 subbank 上，同一条访存指令下每个 subbank 只有一个线程访问，从而消除读读冲突、实现并行读取：

<img src="./images/07_05_simt_transpose/case3_bank.png" alt="case3_bank"  width="1500px" >

由于 padding 只改变 UB 内部的物理布局，不改变转置算法本身，也不改变 GM 读写坐标，因此这个优化只影响 UB 内部的访问效率，不影响计算结果的正确性。多出来的 2 列 padding 不参与计算，只用于让下一行在 UB 中的起始地址错开。

#### 3.5.2 实现思路与代码

核心差异只有 UB 数组定义：


```cpp
constexpr uint32_t tile_pad = 2;
constexpr uint32_t tile_pad_stride = tile_dim + tile_pad;
__ubuf__ float tile[tile_dim][tile_pad_stride];
```

有效数据仍写入前 32 列：

```cpp
tile[local_row][local_col] = input[input_row * width + input_col];
```


转置读和 GM 写回坐标保持不变：

```cpp
output[output_row * height + output_col] = tile[local_col][local_row];
```

因此这个优化只影响 UB 内部访问冲突，不影响结果正确性。完整实现代码与 CMakeLists.txt 已保存在 `Sources/07.05/simt_transpose/ub_padding/` 目录下。

编译运行并采集性能：

In [ ]:
!cd Sources/07.05/simt_transpose/ub_padding && mkdir -p build && cd build && \
 cmake -DCMAKE_ASC_ARCHITECTURES=dav-3510 .. && make -j && \
 ./demo && msopprof ./demo


本实现在 Ascend 950 环境、CANN 9.1.0 上实测的结果如下：

| 阶段 | 核函数 | Task Duration(μs) | aiv_vec_time(μs) |
| --- | --- | --- | --- |
| UB 中转（固定物理核数，1 tile/block） | `transpose_ub_fixed64_custom` | 25.70 | 24.01 |
| UB padding | `transpose_ub_padding_custom` | 16.11 | 14.45 |

`aiv_vec_time` 明显下降（24.01μs → 14.45μs），Task Duration 也随之下降（25.70μs → 16.11μs），说明 padding 确实缓解了转置方向读取 UB 时的 bank 冲突，SIMT转置读取的效率明显提升。不过 `aiv_vec_time` 仍然占了 Task Duration 的绝大部分，说明 UB 内部访问和同步开销依然是主要瓶颈之一。下一步引入双缓冲，让相邻迭代之间的 GM 访问与 UB 内部计算重叠执行。

### 3.6 双缓冲流水并行

#### 3.6.1 为什么需要双缓冲：循环尾部同步的代价

`ub_padding` 实现中，每次循环迭代都要经历"加载 tile → `asc_syncthreads()` → 转置写回 → `asc_syncthreads()`"，这两次同步分别防止两种数据冲突：

- 第一次同步（加载之后）：确保 Thread Block 内所有线程都已经把 tile 数据写入 UB，才能开始按转置方向读取，否则可能读到还未写入的数据。
- 第二次同步（写回之后）：确保当前 tile 的转置读取全部完成，才能进入下一轮循环开始加载新数据，否则下一轮的加载会覆盖还在被读取的 UB 内容。

第二次同步（循环尾部同步）导致的直接后果是：当前迭代的转置写回必须全部完成后，下一迭代的地址计算和 tile 加载才能开始，两者之间不存在任何重叠，GM 写和 GM 读被强制串行。
`ub_padding` 实现的仿真指令流水图也揭示了整个流程串行的效果：

<img src="./images/07_05_simt_transpose/case6_trace.png" alt="case6_trace"  width="1500px" >

其中耗时最多的SIMT_LDG和SIMT_STG分别为SIMT编程模式下从GM读取数据和向GM写入数据的指令，每轮循环受尾部 `asc_syncthreads()` 约束，当前轮GM写入阶段完成后才能进入下一轮循环，下一轮数据加载前的地址计算与GM读取阶段需要串行执行。

双缓冲（ping/pong）通过引入两份 UB 缓冲区来去掉这次尾部同步：用一个在 0/1 之间切换的下标选择当前使用哪一份缓冲区，当前迭代的转置写回使用其中一份缓冲区的数据，下一迭代的加载则写入另一份缓冲区，不需要等待写回完成即可开始，两者因此可以并行执行。

#### 3.6.2 实现思路与代码

把 UB tile 数组扩展为两份，并用一个在 0/1 之间切换的下标 `cnt` 选择当前使用哪一份：

```cpp

__ubuf__ float tile[2][tile_dim][tile_pad_stride];
uint32_t cnt = 0;

for (uint32_t tile_id = blockIdx.x; tile_id < total_tiles; tile_id += gridDim.x) {
    tile[cnt][local_row][local_col] = input[input_row * width + input_col];
    asc_syncthreads(); // 只保留数据加载后的同步

    output[output_row * height + output_col] = tile[cnt][local_col][local_row];

    cnt ^= 1; // 切换到下一份缓冲区
}
```

相比 `ub_padding`，这里去掉了循环尾部（转置写回之后）的 `asc_syncthreads()`，只保留了数据加载之后的同步。加载后的同步不能去掉：转置读取仍然依赖本轮 Thread Block 内所有线程都已经把数据写入当前这份缓冲区。但去掉尾部同步之后，当前迭代的转置写回和下一迭代的 tile 加载、地址计算可以在不同的缓冲区上并行进行：下一轮加载写入的是 `tile[cnt ^ 1]`，而当前轮写回读取的是 `tile[cnt]`，两者互不干扰。只有当某一轮真正要读取自己那份缓冲区时（即经过两轮之后再次轮回到同一份缓冲区），才需要等待对应的加载完成——这个等待已经由加载后保留的 `asc_syncthreads()` 隐式保证。

#### 3.6.3 编译运行并采集性能

编译运行双缓冲实现：

In [ ]:
!cd Sources/07.05/simt_transpose/ub_padding_db && mkdir -p build && cd build && \
 cmake -DCMAKE_ASC_ARCHITECTURES=dav-3510 .. && make -j && \
 ./demo


编译运行成功后，若看到以下输出，则说明计算结果与预期完全一致：

```text
[Success] Case accuracy verification passed.
```

完成正确性验证后，使用 `msOpProf` 工具采集算子性能：


In [ ]:
!cd Sources/07.05/simt_transpose/ub_padding_db/build && msopprof ./demo


本实现在 Ascend 950 环境、CANN 9.1.0 上实测的结果如下：

| 阶段 | 核函数 | Task Duration(μs) | aiv_vec_time(μs) |
| --- | --- | --- | --- |
| UB padding | `transpose_ub_padding_custom` | 16.11 | 14.45 |
| UB padding + 双缓冲 | `transpose_ub_padding_db_custom` | 12.49 | 10.81 |

去掉循环尾部同步、引入双缓冲后，Task Duration 进一步下降（16.11μs → 12.49μs），`aiv_vec_time` 同步下降（14.45μs → 10.81μs）。仿真流水图如下：

<img src="./images/07_05_simt_transpose/case7_trace.png" alt="case7_trace"  width="1500px" >

可以明显看到，相邻迭代之间访存和计算的重叠，当前迭代的转置写回不再阻塞下一迭代的 tile 加载和地址计算，流水间的耗时掩盖有效减少了整体耗时。

至此，本节的转置优化路径已经全部完成：从最初直接读写 GM 的实现（57.37μs）优化到双缓冲版本（12.49μs），降到初始版本的约 22%。为了衡量这个最终结果距离硬件访存效率的理论上限还有多远，下面补充一个不做任何转置计算的纯连续读写基线，作为整条优化路径的最终参照。

#### 3.6.4 基线对比：GM 带宽基线（连续拷贝）

前面的优化路径已经把 Task Duration 从 57.37μs 压缩到 12.49μs，但这个数值本身并不能说明访存效率是否已经逼近硬件上限。为此，补充一个不做任何转置计算的纯连续拷贝核函数，摸清当前硬件在最理想访存模式下的 GM 带宽上限，作为整条优化路径的最终参照。这个基线核函数把输入、输出矩阵都当作一维数组，每个线程按线性地址连续读写，不做任何坐标变换：

```cpp
uint32_t idx = blockIdx.x * blockDim.x + threadIdx.x;
uint32_t stride = gridDim.x * blockDim.x;
for (uint32_t i = idx; i < elements; i += stride) {
    output[i] = input[i];
}
```

这个实现的访存模式是 `GM 连续读 + GM 连续写`，是所有转置实现理论上能够达到的性能上限参照。如果某个转置实现的 Task Duration 已经接近这个基线，说明访存效率已经很难再继续提升，后续需要转向计算侧的优化。完整实现代码与 CMakeLists.txt 已保存在 `Sources/07.05/simt_transpose/copy_baseline/` 目录下。

#### 3.6.5 编译运行并采集性能

CMakeLists.txt 已保存在 `Sources/07.05/simt_transpose/copy_baseline/` 目录下，执行以下命令编译并运行当前实现：

In [ ]:
!cd Sources/07.05/simt_transpose/copy_baseline && mkdir -p build && cd build && \
 cmake -DCMAKE_ASC_ARCHITECTURES=dav-3510 .. && make -j && \
 ./demo

编译运行成功后，若看到以下输出，则说明计算结果与预期完全一致：

```text
[Success] Case accuracy verification passed.
```

完成正确性验证后，使用 `msOpProf` 工具采集算子性能：

In [ ]:
!cd Sources/07.05/simt_transpose/copy_baseline/build && msopprof ./demo

本实现在 Ascend 950 环境、CANN 9.1.0 上实测的结果如下（单次采集结果会受设备状态影响，实际数值以本机采集为准）：

| 阶段 | 核函数 | Task Duration(μs) | aiv_vec_time(μs) | aiv_scalar_time(μs) |
| --- | --- | --- | --- | --- |
| UB padding + 双缓冲 | `transpose_ub_padding_db_custom` | 12.49 | 10.81 | -- |
| GM 带宽基线 | `copy_custom` | 6.26 | 4.58 | 0.42 |

这个数值反映了当前硬件在纯连续读写场景下的 GM 访存效率上限。对比最终的双缓冲实现（12.49μs）和这个基线（6.26μs）可以看到，即便经过完整的优化路径，实际带宽达到理论带宽的50%，也说明如果要继续压缩耗时，需要从减少同步次数、进一步提升数据复用等计算侧手段入手，而不是单纯依赖访存模式优化。

## 4. 小结

本节完整走完了 SIMT Transpose 算子的逐步优化路径，每一步都通过真实编译运行采集了性能数据，并在最后补充了一个纯连续拷贝的 GM 带宽基线作为参照：

| 阶段 | 核函数 | Task Duration(μs) | Block Dim | 本步引入的变化 |
| --- | --- | --- | --- | --- |
| 直接读写 GM | `transpose_gm_custom` | 57.37 | 512 | 每线程处理一个元素，转置写回跨行导致 GM 写非连续，且 Thread Block 超发 |
| 直接读写 GM（限制核数） | `transpose_gm_core_limited_custom` | 35.674 | 64 | 消除 Thread Block 超发，但非连续写开销从调度排队中暴露出来 |
| UB 中转（固定核数，2 tile/block） | `transpose_ub_2tile_core_limited_custom` | 27.08 | 64 | 引入 UB 中转，GM 读写恢复连续，但寄存器溢出 |
| UB 中转（固定核数，限制最大线程数） | `transpose_ub_fixed64_custom` | 25.70 | 64 | 降低每 Thread Block 线程数，消除寄存器溢出 |
| UB padding | `transpose_ub_padding_custom` | 16.11 | 64 | `32x34` padding，消除转置读 bank 冲突 |
| UB padding + 双缓冲 | `transpose_ub_padding_db_custom` | 12.49 | 64 | ping/pong 双缓冲，重叠相邻迭代的访存与计算 |

从这条路径中可以总结出几条通用的 SIMT 算子调优思路：

- **Thread Block 数量不要超过物理核数**：按"任务量直接决定 Thread Block 数"的朴素思路启动核函数（每线程处理一个元素），实现起来最简单，但 Thread Block 数容易远超物理核数，超发部分需要排队调度，产生额外的头尾开销。
- **优先把非连续访问转移到访问效率更高的存储层级**：直接在 GM 上非连续读写的问题会造成较大的访存时延，可以考虑引入 UB 中转，将非连续访问限制在 UB 内部，GM 读写保持连续，能显著提升访存效率。
- **计算复杂度较高时要关注寄存器资源，限制合适的最大线程数避免寄存器溢出**：简单粗暴地把 Thread Block 数改成物理核数、维持原来的线程数不变，可能因为单个 Thread Block 要处理的数据和索引变量增多导致寄存器溢出，抵消调度层面的收益。要结合编译日志中的 `Stack size`、`Used register number` 等信息，找到线程数和寄存器压力之间的平衡点。
- **关注片上存储（UB）内部的访问冲突**：即使数据已经搬进 UB，转置这种按列访问的模式仍可能因为 bank 冲突降低访问效率，padding 是一种常见且低成本的缓解手段。
- **用双缓冲重叠流水线各阶段**：当循环体内存在"当前迭代结果用完之前，下一迭代无法开始"的强同步点时，双缓冲可以让不同迭代的访存和计算相互重叠，进一步压缩整体耗时。
- **用一个纯连续读写的基线衡量优化空间**：在完成具体算子的优化后，可以用一个不含业务逻辑的纯连续读写核函数摸清硬件访存效率的上限，衡量当前实现距离这个上限还有多远，判断是否还有进一步优化的空间，以及后续应该往访存侧还是计算侧发力。

需要说明的是，本节展示的性能数据全部来自Ascend 950 环境、CANN 9.1.0 版本实测，实际数值会受设备状态、驱动版本等因素影响而波动，但各阶段之间的相对趋势（GM 直接转置最慢、UB 中转后大幅提升、Thread Block 数量与寄存器压力需要权衡、padding 和双缓冲进一步优化、纯连续拷贝基线远快于所有转置实现）具有普遍参考意义。

Transpose 的非连续访问，本质上来自"转置"这个操作本身：输入按行连续，输出却要按列连续，这个矛盾没法通过换一种实现方式消除，只能像本节这样把非连续访问转移到 UB 内部。下面的课后练习换一个角度：来看一个输入、输出在 GM 上都能保持连续访问的算子——MaxPool，练习如何在没有非连续访问负担的前提下，把双缓冲这项技巧用到位。

## 课后编程习题：MaxPool 算子的双缓冲优化

**算子语义**：MaxPool（最大池化）是一种常见的下采样操作，把输入矩阵划分成互不重叠的窗口，每个窗口内取最大值作为输出。本习题使用 `4 x 4` 窗口、步长为 `4`（不重叠），计算公式为：

```text
output(i, j) = max(input[4i:4i+4, 4j:4j+4])
```

也就是说，输出矩阵中的每个元素，是输入矩阵中对应 `4 x 4` 窗口内 16 个元素的最大值。由于窗口不重叠，且 `1024 / 4` 能整除，不需要考虑边界 padding。

**规格：**

| 项 | 取值 |
| --- | --- |
| 核函数名 | `maxpool_ub_db_custom` |
| 输入 `input` | `(1024, 1024)`，`float` |
| 输出 `output` | `(256, 256)`，`float` |
| 窗口/步长 | `4 x 4`，不重叠 |
| Thread Block 数 | 物理核数（`get_vector_core_num()` 运行时查询，测试机器上为 64） |
| 每 Block 线程数 | 256（对应一个 band 的输出宽度） |

和 Transpose 不同，MaxPool 的输入、输出在 GM 上都是连续访问：把输入按每 `4` 行分成一个 band（band 大小为 `4 x 1024`），一个 band 经过池化后恰好对应输出矩阵的一整行（`256` 个元素）。band 内读取、band 输出写回都不需要跨行跳跃，因此本习题不需要像 Transpose 那样先解决非连续访问的问题，可以把精力完全放在双缓冲流水线的搭建上。

**要求**：请参照本节 3.6 节 `transpose_ub_padding_db_custom` 的双缓冲结构（`__ubuf__` 双份缓冲区 + `cnt ^= 1` 轮换 + 只保留数据加载后的 `asc_syncthreads()`），自己实现纯 SIMT 版本的 MaxPool 双缓冲优化：

1. 每个 Thread Block 通过 grid-stride 循环处理多个 band（band 数量共 `256` 个，等于总行数 `1024` 除以窗口高度 `4`）。
2. 每次循环中，本 Thread Block 的每个线程（`local_col = threadIdx.x`）直接从 GM 读取自己负责的一列 `4 x 4` 窗口（`4` 行、每行 `4` 个元素），写入当前缓冲区 `band[cnt]`。
3. 用 `asc_syncthreads()` 确保整个 band 都加载完成后，再对 `band[cnt]` 中本线程负责的 `4 x 4` 窗口做最大值规约（用三目比较逐元素累积，不要使用 `std::max`/`fmaxf`），并把结果直接写回 GM 对应位置。
4. 参照 3.6 节的做法，去掉循环尾部同步，只保留加载后的同步，让当前迭代的规约写回和下一迭代的加载在两份缓冲区上重叠执行。

下面的代码骨架已经搭好双缓冲的循环结构（缓冲区声明、`cnt` 轮换、`asc_syncthreads()` 调用位置），需要你补全两处 `TODO`：把数据从 GM 加载到 `band[cnt]`，以及对 `band[cnt]` 做窗口最大值规约并写回 `output`。

In [ ]:
!mkdir -p Sources/07.05/simt_maxpool

执行以下操作先将骨架代码写入 `Sources/07.05/simt_maxpool/simt_maxpool.asc`。请补全两处 `TODO`，然后执行本 cell 完成写入（源码同时保留在 `src/simt_maxpool/simt_maxpool.asc`，可作为对照）：

In [ ]:
%%writefile Sources/07.05/simt_maxpool/simt_maxpool.asc
#include <algorithm>
#include <cmath>
#include <iostream>
#include <iterator>
#include <vector>
#include "acl/acl.h"
#include "simt_api/asc_simt.h"

constexpr uint32_t WINDOW = 4; // 池化窗口/步长，4x4 不重叠
constexpr uint32_t MAX_THREAD_COUNT = 1024;

// 纯 SIMT 核函数：band（WINDOW 行）级别双缓冲，每线程负责一个输出列的 4x4 窗口最大值
template <uint32_t window>
__global__ __launch_bounds__(MAX_THREAD_COUNT) void maxpool_ub_db_custom(
    float* output, const float* input, uint32_t width, uint32_t height, uint32_t total_bands)
{
    uint32_t out_width = width / window;
    __ubuf__ float band[2][window][1024]; // 双缓冲：轮换缓冲区，去除尾部同步

    uint32_t local_col = threadIdx.x;
    uint32_t cnt = 0;

    for (uint32_t band_id = blockIdx.x; band_id < total_bands; band_id += gridDim.x) {
        // TODO: 把 band_id 对应的 4 行、width 列输入，按每个线程负责一列 4x4 窗口的方式读入 band[cnt]
        asc_syncthreads(); // 只保留数据加载后的同步

        // TODO: 对 band[cnt] 中本线程负责的 4x4 窗口做最大值规约，写入 output[band_id * out_width + local_col]

        cnt ^= 1; // 切换到下一份缓冲区
    }
}

// Host 侧 golden 函数：output(i, j) = max(input[4i:4i+4, 4j:4j+4])
void maxpool_golden(
    const std::vector<float>& input, std::vector<float>& golden, uint32_t width, uint32_t height, uint32_t window)
{
    uint32_t out_width = width / window;
    uint32_t out_height = height / window;
    for (uint32_t i = 0; i < out_height; ++i) {
        for (uint32_t j = 0; j < out_width; ++j) {
            float max_val = input[(i * window) * width + j * window];
            for (uint32_t r = 0; r < window; ++r) {
                for (uint32_t c = 0; c < window; ++c) {
                    float v = input[(i * window + r) * width + j * window + c];
                    max_val = (v > max_val) ? v : max_val;
                }
            }
            golden[i * out_width + j] = max_val;
        }
    }
}

// 结果校验函数
uint32_t verify_result(std::vector<float>& output, std::vector<float>& golden)
{
    auto print_tensor = [](std::vector<float>& tensor, const char* name) {
        constexpr size_t max_print_size = 20;
        std::cout << name << ": ";
        std::copy(
            tensor.begin(), tensor.begin() + std::min(tensor.size(), max_print_size),
            std::ostream_iterator<float>(std::cout, " "));
        if (tensor.size() > max_print_size) {
            std::cout << "...";
        }
        std::cout << std::endl;
    };
    print_tensor(output, "Output");
    print_tensor(golden, "Golden");
    for (size_t i = 0; i < output.size(); ++i) {
        if (std::fabs(output[i] - golden[i]) > 1e-3f) {
            std::cout << "[Failed] Case accuracy verification failed!" << std::endl;
            return 1;
        }
    }
    std::cout << "[Success] Case accuracy verification passed." << std::endl;
    return 0;
}

// 查询硬件 vector core 数（AIV 核数），不同环境上的物理核数不同，需要在运行时获取
uint32_t get_vector_core_num(uint32_t device_id)
{
    int64_t core_num = 0;
    aclError ret = aclrtGetDeviceInfo(device_id, ACL_DEV_ATTR_VECTOR_CORE_NUM, &core_num);
    if (ret != ACL_SUCCESS) {
        return 0;
    }
    return static_cast<uint32_t>(core_num);
}

int32_t main(int32_t argc, char* argv[])
{
    constexpr uint32_t in_height = 1024;
    constexpr uint32_t in_width = 1024;
    constexpr uint32_t in_total_length = in_height * in_width;
    constexpr uint32_t out_width = in_width / WINDOW;
    constexpr uint32_t out_height = in_height / WINDOW;
    constexpr uint32_t out_total_length = out_width * out_height;
    constexpr uint32_t total_bands = in_height / WINDOW;

    // 构造输入数据
    std::vector<float> input(in_total_length);
    for (uint32_t i = 0; i < in_total_length; ++i) {
        input[i] = static_cast<float>((i % 1000) * 1.25f);
    }

    // 计算预期结果
    std::vector<float> golden(out_total_length);
    maxpool_golden(input, golden, in_width, in_height, WINDOW);

    constexpr size_t in_byte_size = in_total_length * sizeof(float);
    constexpr size_t out_byte_size = out_total_length * sizeof(float);

    // 初始化与创建 stream
    aclInit(nullptr);
    int32_t device_id = 0;
    aclrtSetDevice(device_id);

    // Thread Block 数固定为物理核数；需在 aclrtSetDevice 之后查询
    uint32_t num_blocks = get_vector_core_num(device_id);
    if (num_blocks == 0) {
        std::cout << "[Failed] Get vector core num failed!" << std::endl;
        return 1;
    }
    std::cout << "Vector core num: " << num_blocks << std::endl;
    aclrtStream stream = nullptr;
    aclrtCreateStream(&stream);

    float* input_device = nullptr;
    float* output_device = nullptr;
    uint8_t* output_host = nullptr;

    // 分配 Host / Device 内存并把数据从 Host 拷贝到 Device
    aclrtMalloc((void**)&input_device, in_byte_size, ACL_MEM_MALLOC_HUGE_FIRST);
    aclrtMemcpy(input_device, in_byte_size, input.data(), in_byte_size, ACL_MEMCPY_HOST_TO_DEVICE);

    aclrtMalloc((void**)&output_device, out_byte_size, ACL_MEM_MALLOC_HUGE_FIRST);
    aclrtMallocHost((void**)(&output_host), out_byte_size);

    // 启动核函数
    dim3 grid(num_blocks, 1, 1);
    dim3 block(out_width, 1, 1);
    maxpool_ub_db_custom<WINDOW><<<grid, block, 0, stream>>>(
        output_device, input_device, in_width, in_height, total_bands);

    // 同步等待核函数执行完成
    aclrtSynchronizeStream(stream);

    // 把结果从 Device 拷回 Host
    aclrtMemcpy(output_host, out_byte_size, output_device, out_byte_size, ACL_MEMCPY_DEVICE_TO_HOST);
    std::vector<float> result((float*)output_host, (float*)(output_host + out_byte_size));

    // 释放内存
    aclrtFree(input_device);
    aclrtFree(output_device);
    aclrtFreeHost(output_host);

    // 去初始化
    aclrtDestroyStream(stream);
    aclrtResetDevice(device_id);
    aclFinalize();

    // 校验结果
    return verify_result(result, golden);
}

对应的 `CMakeLists.txt` 同样写入工作目录：

In [ ]:
%%writefile Sources/07.05/simt_maxpool/CMakeLists.txt
cmake_minimum_required(VERSION 3.16)

set(CMAKE_ASC_ARCHITECTURES "dav-3510" CACHE STRING "NPU ARCH, e.g. dav-3510")

find_package(ASC REQUIRED)
project(simt_maxpool_sample LANGUAGES ASC CXX)

add_executable(demo
    simt_maxpool.asc
)

target_compile_options(demo PRIVATE
    $<$<COMPILE_LANGUAGE:ASC>:--npu-arch=${CMAKE_ASC_ARCHITECTURES} --enable-simt>
)

In [ ]:
!cd Sources/07.05/simt_maxpool && mkdir -p build && cd build && \
 cmake -DCMAKE_ASC_ARCHITECTURES=dav-3510 .. && make -j && \
 ./demo


请先在上面的 `%%writefile` cell 中补全 TODO 并执行该 cell 完成写入，再执行编译单元格。编译运行成功后，若看到以下输出，则说明你补全的两处 TODO（数据加载、窗口最大值规约与写回）逻辑正确：

```text
[Success] Case accuracy verification passed.
```

如果验证失败，可以对照 3.6 节 `transpose_ub_padding_db_custom` 的双缓冲结构，重点检查两点：加载阶段是否严格按 `local_col` 对应的 `4 x 4` 窗口读取（而不是搬入整行数据）；规约阶段是否读取的是 `band[cnt]` 中当前缓冲区（而不是 `band[cnt ^ 1]`）。

**参考答案：**

In [ ]:
!cat answer/07_05_simt_maxpool/simt_maxpool.asc

**小结：Pool 与 Transpose 的访存对比**

回顾本节 Transpose 的优化路径，最终耗时的下降主要来自两类手段：把非连续访问从 GM 转移到 UB 内部（UB 中转），以及用双缓冲重叠相邻迭代的访存与计算。MaxPool 练习让你看到同样的"双缓冲"技巧，用在一个访存模式完全不同的算子上：

- **Transpose**：输入按行连续，输出必须按列连续写回，这个矛盾来自"转置"操作本身，无法通过换一种实现方式消除，只能把非连续访问转移到 UB 内部（3.4~3.5 节），双缓冲（3.6 节）是在此基础上进一步压缩流水线尾部同步开销。
- **MaxPool**：窗口 `4x4` 不重叠，使得输入按 band 连续读、输出按行连续写天然成立，从第一步实现开始就不存在非连续访问问题，因此主要考虑使用双缓冲实现流水并行的优化手段。

也就是说，同一个双缓冲技巧，在 Transpose 里是解决完非连续访问问题之后的"最后一步优化"，在 MaxPool 里则是唯一需要关心的优化点——这也是本习题选择 MaxPool 作为练习对象的原因：让你在没有非连续访问负担的情况下，单独巩固双缓冲流水线的搭建能力。